In [1]:
import os, time, random, numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# -------------------- Setup --------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -------------------- Data Loading --------------------
def load_dataset(data_path, dataset_name, config):
    file_path = os.path.join(data_path, config['file'])
    data = sio.loadmat(file_path)[config['data_key']]
    
    if config['file'] == config['gt_file']:
        labels = sio.loadmat(file_path)[config['gt_key']]
    else:
        gt_path = os.path.join(data_path, config['gt_file'])
        labels = sio.loadmat(gt_path)[config['gt_key']]
    
    if len(labels.shape) == 3:
        labels = labels.squeeze()
    return data, labels

def applyPCA(X, numComponents):
    original_shape = X.shape
    X_reshaped = X.reshape(-1, X.shape[-1])
    pca = PCA(n_components=min(numComponents, X.shape[-1]), whiten=True)
    X_pca = pca.fit_transform(X_reshaped)
    return X_pca.reshape(original_shape[0], original_shape[1], -1)

def createImageCubes(X, y, windowSize=5):
    margin = windowSize // 2
    paddedX = np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='reflect')
    patches, labels_ = [], []
    for r in range(margin, paddedX.shape[0] - margin):
        for c in range(margin, paddedX.shape[1] - margin):
            if y[r - margin, c - margin] > 0:
                patch = paddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]
                patches.append(patch)
                labels_.append(y[r - margin, c - margin] - 1)
    return np.array(patches), np.array(labels_)

# -------------------- Self‑Adaptive Clustering --------------------
class AdaptiveClustering(nn.Module):
    def __init__(self, weight_shape):
        super().__init__()
        self.out_channels = weight_shape[0]
        self.D = int(np.prod(weight_shape[1:]))
        self.keep_ratio = nn.Parameter(torch.tensor(0.5))
        self.tau = nn.Parameter(torch.tensor(1.0))

    @property
    def K(self):
        k = int(max(2, torch.sigmoid(self.keep_ratio).item() * self.out_channels))
        return min(k, self.out_channels)

    def get_centroids(self):
        if not hasattr(self, '_centroids_buffer') or self._centroids_buffer.shape[0] != self.K:
            self.register_buffer('_centroids_buffer', torch.randn(self.K, self.D) * 0.01)
        return self._centroids_buffer

    def forward(self, W):
        W_flat = W.view(W.shape[0], -1)
        O, D = W_flat.shape
        K_eff = self.K
        centroids = self.get_centroids()[:K_eff]
        tau_val = self.tau.abs() + 0.05
        dist = torch.cdist(W_flat, centroids)
        A = F.softmax(-dist / tau_val, dim=1)
        W_compressed = torch.mm(A, centroids)
        return W_compressed.view_as(W), A

# -------------------- Fully Self‑Adaptive HybridSN --------------------
class FullySelfAdaptiveHybridSN(nn.Module):
    def __init__(self, n_classes, S, L, compress_layers=None, gate_init=0.0):
        super().__init__()
        self.compress_layers = compress_layers if compress_layers is not None else []
        self.compression_modules = nn.ModuleDict()
        self.gates = nn.ParameterDict()
        self.S, self.L = S, L

        self.conv3d_1 = nn.Conv3d(1, 8, kernel_size=(3,3,7))
        self.conv3d_2 = nn.Conv3d(8, 16, kernel_size=(1,1,5))
        self.conv3d_3 = nn.Conv3d(16, 32, kernel_size=(1,1,3))

        with torch.no_grad():
            dummy = torch.zeros(1, 1, S, S, L)
            dummy = F.relu(self.conv3d_1(dummy))
            dummy = F.relu(F.conv3d(dummy, self.conv3d_2.weight, self.conv3d_2.bias, padding=(1,1,2)))
            dummy = F.relu(F.conv3d(dummy, self.conv3d_3.weight, self.conv3d_3.bias, padding=(1,1,1)))
            _, C, H, W, D = dummy.shape
            self.conv2d = nn.Conv2d(C * D, 64, kernel_size=1)
            dummy = F.relu(self.conv2d(dummy.view(1, C * D, H, W)))
            self.flatten_size = dummy.numel()

        self.fc1 = nn.Linear(self.flatten_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, n_classes)
        self.drop1 = nn.Dropout(0.4)
        self.drop2 = nn.Dropout(0.4)

        layer_map = {
            'conv3d_1': self.conv3d_1, 'conv3d_2': self.conv3d_2, 'conv3d_3': self.conv3d_3,
            'conv2d': self.conv2d, 'fc1': self.fc1, 'fc2': self.fc2
        }
        for name in compress_layers:
            if name in layer_map:
                layer = layer_map[name]
                self.compression_modules[name] = AdaptiveClustering(layer.weight.shape)
                self.gates[name] = nn.Parameter(torch.tensor(gate_init))

    def to(self, device):
        super().to(device)
        return self

    def get_gate(self, name):
        if self.gates is None or name not in self.gates:
            return 0.0
        return torch.sigmoid(self.gates[name])

    def get_compressed_weight(self, name, original_weight):
        if self.compression_modules is None or name not in self.compression_modules:
            return original_weight
        gate = self.get_gate(name)
        if gate < 0.01:
            return original_weight
        compressed, _ = self.compression_modules[name](original_weight)
        return (1 - gate) * original_weight + gate * compressed

    def forward(self, x):
        x = x.unsqueeze(1)
        
        w1 = self.get_compressed_weight('conv3d_1', self.conv3d_1.weight)
        x = F.relu(F.conv3d(x, w1, self.conv3d_1.bias))
        
        w2 = self.get_compressed_weight('conv3d_2', self.conv3d_2.weight)
        x = F.relu(F.conv3d(x, w2, self.conv3d_2.bias, padding=(1,1,2)))
        
        w3 = self.get_compressed_weight('conv3d_3', self.conv3d_3.weight)
        x = F.relu(F.conv3d(x, w3, self.conv3d_3.bias, padding=(1,1,1)))
        
        B, C, H, W, D = x.shape
        x = x.view(B, C * D, H, W)

        w_conv2d = self.get_compressed_weight('conv2d', self.conv2d.weight)
        x = F.relu(F.conv2d(x, w_conv2d, self.conv2d.bias))
        x = torch.flatten(x, 1)

        w_fc1 = self.get_compressed_weight('fc1', self.fc1.weight)
        x = F.relu(F.linear(x, w_fc1, self.fc1.bias))
        x = self.drop1(x)

        w_fc2 = self.get_compressed_weight('fc2', self.fc2.weight)
        x = F.relu(F.linear(x, w_fc2, self.fc2.bias))
        x = self.drop2(x)

        return self.fc3(x)

    def sparsity_loss(self, lambda_base=1.5):
        if self.gates is None or len(self.gates) == 0:
            return torch.tensor(0.0)
        loss = 0.0
        for name, gate_param in self.gates.items():
            gate = self.get_gate(name)
            loss += lambda_base * (1 - gate)
            if name in self.compression_modules:
                keep = torch.sigmoid(self.compression_modules[name].keep_ratio)
                loss += 0.8 * keep
        return loss / len(self.gates)

    def get_learned_K(self):
        """Get learned K values for all compressible layers"""
        k_vals = {}
        if self.compression_modules:
            for name in self.compression_modules:
                k_vals[name] = self.compression_modules[name].K
        return k_vals

    def finalize_compression(self, gate_threshold=0.4):
        if self.compression_modules is None:
            return
        new_sizes = {}
        
        for name in list(self.compression_modules.keys()):
            gate = self.get_gate(name).item()
            if gate > gate_threshold:
                mod = self.compression_modules[name]
                K_eff = mod.K
                layer = getattr(self, name)
                orig_out = layer.out_channels if hasattr(layer, 'out_channels') else layer.out_features
                print(f"  Compressing {name}: {orig_out} → {K_eff}")
                
                with torch.no_grad():
                    compressed_weight = layer.weight.data[:K_eff]
                    compressed_bias = layer.bias.data[:K_eff] if layer.bias is not None else None
                    
                    if isinstance(layer, nn.Conv3d):
                        new_layer = nn.Conv3d(layer.in_channels, K_eff, kernel_size=layer.kernel_size,
                                             stride=layer.stride, padding=layer.padding)
                        new_layer.weight.data = compressed_weight.view(K_eff, layer.in_channels, *layer.kernel_size)
                        if compressed_bias is not None:
                            new_layer.bias.data = compressed_bias
                    elif isinstance(layer, nn.Conv2d):
                        new_layer = nn.Conv2d(layer.in_channels, K_eff, kernel_size=layer.kernel_size, padding=layer.padding)
                        new_layer.weight.data = compressed_weight.view(K_eff, layer.in_channels, 1, 1)
                        if compressed_bias is not None:
                            new_layer.bias.data = compressed_bias
                    elif isinstance(layer, nn.Linear):
                        new_layer = nn.Linear(layer.in_features, K_eff)
                        new_layer.weight.data = compressed_weight
                        if compressed_bias is not None:
                            new_layer.bias.data = compressed_bias
                    else:
                        continue
                    setattr(self, name, new_layer)
                    new_sizes[name] = K_eff
            else:
                print(f"  Keeping {name}: gate={gate:.3f} < {gate_threshold}")

        self._update_dependent_dimensions(new_sizes)
        self.compression_modules = nn.ModuleDict()
        self.gates = nn.ParameterDict()
        print("✅ Compression finalized")

    def _update_dependent_dimensions(self, new_sizes):
        if 'conv3d_1' in new_sizes:
            old = self.conv3d_2
            new = nn.Conv3d(new_sizes['conv3d_1'], old.out_channels, kernel_size=old.kernel_size, padding=old.padding)
            with torch.no_grad():
                new.weight.data = old.weight.data[:, :new_sizes['conv3d_1'], :, :, :]
                new.bias.data = old.bias.data
            self.conv3d_2 = new
        
        if 'conv3d_2' in new_sizes:
            old = self.conv3d_3
            new = nn.Conv3d(new_sizes['conv3d_2'], old.out_channels, kernel_size=old.kernel_size, padding=old.padding)
            with torch.no_grad():
                new.weight.data = old.weight.data[:, :new_sizes['conv3d_2'], :, :, :]
                new.bias.data = old.bias.data
            self.conv3d_3 = new
        
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.S, self.S, self.L)
            dummy = F.relu(self.conv3d_1(dummy))
            dummy = F.relu(F.conv3d(dummy, self.conv3d_2.weight, self.conv3d_2.bias, padding=(1,1,2)))
            dummy = F.relu(F.conv3d(dummy, self.conv3d_3.weight, self.conv3d_3.bias, padding=(1,1,1)))
            _, C, H, W, D = dummy.shape
            conv3d_out = C * D
        
        if conv3d_out != self.conv2d.in_channels:
            old = self.conv2d
            new = nn.Conv2d(conv3d_out, old.out_channels, kernel_size=1)
            with torch.no_grad():
                new.weight.data = old.weight.data[:, :conv3d_out, :, :]
                new.bias.data = old.bias.data
            self.conv2d = new
        
        with torch.no_grad():
            dummy = dummy.view(1, conv3d_out, H, W)
            dummy = F.relu(self.conv2d(dummy))
            new_flatten = dummy.numel()
        
        if new_flatten != self.fc1.in_features:
            old = self.fc1
            new = nn.Linear(new_flatten, old.out_features)
            with torch.no_grad():
                new.weight.data = old.weight.data[:, :new_flatten]
                new.bias.data = old.bias.data
            self.fc1 = new
        
        if 'fc1' in new_sizes:
            old = self.fc2
            new = nn.Linear(new_sizes['fc1'], old.out_features)
            with torch.no_grad():
                new.weight.data = old.weight.data[:, :new_sizes['fc1']]
                new.bias.data = old.bias.data
            self.fc2 = new
        
        if 'fc2' in new_sizes:
            old = self.fc3
            new = nn.Linear(new_sizes['fc2'], old.out_features)
            with torch.no_grad():
                new.weight.data = old.weight.data[:, :new_sizes['fc2']]
                new.bias.data = old.bias.data
            self.fc3 = new

# -------------------- Training Functions --------------------
def train_two_stage(model, train_loader, X_test_t, y_test_t,
                    epochs_stage1=150, epochs_stage2=60, final_lambda=1.8, lr=0.0005):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    ce_loss = nn.CrossEntropyLoss()
    
    # Stage 1
    print("\n--- Stage 1: Training without compression loss ---")
    best_val_acc = 0
    best_state = None
    patience = 30
    no_improve = 0
    
    for epoch in range(epochs_stage1):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            out_val = model(X_test_t)
            val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience and best_val_acc > 0.98:
                print(f"Early stopping at epoch {epoch+1} (best val_acc={best_val_acc:.4f})")
                model.load_state_dict(best_state)
                break
        
        if (epoch+1) % 20 == 0:
            print(f"Stage1 Epoch {epoch+1}/{epochs_stage1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    # Stage 2 with K evolution tracking
    print("\n--- Stage 2: Adding compression loss (K evolution will be shown) ---")
    for epoch in range(epochs_stage2):
        model.train()
        total_loss = 0.0
        lam = final_lambda * (epoch / max(1, epochs_stage2-1))
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss += lam * model.sparsity_loss()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch+1) % 10 == 0 or epoch == epochs_stage2-1:
            model.eval()
            with torch.no_grad():
                out_val = model(X_test_t)
                val_acc = accuracy_score(y_test_t.cpu(), out_val.argmax(1).cpu())
            
            gates = {name: model.get_gate(name).item() for name in model.gates}
            k_vals = model.get_learned_K()
            
            print(f"\n📊 Epoch {epoch+1}/{epochs_stage2} | λ={lam:.3f} | Val Acc={val_acc:.4f}")
            print(f"   Gates: {gates}")
            print(f"   Learned K: {k_vals}")
    
    return model

def evaluate_full(model, X, y_true):
    model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        out = model(X)
    t2 = time.perf_counter()
    inference_time_ms = (t2 - t1) * 1000
    
    pred = out.argmax(dim=1).cpu().numpy()
    y_true = y_true.cpu().numpy()
    
    return {
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, pred, average='macro', zero_division=0),
        'inference_time_ms': inference_time_ms
    }

# -------------------- Main Experiment --------------------
def run_experiment_on_dataset(data_path, dataset_name, config, ep_baseline=100, ep_stage2=50, ep_fine=80):
    print("\n" + "="*80)
    print(f"DATASET: {dataset_name}")
    print(f"Description: {config['description']}")
    print(f"PCA Components: {config['pca_components']}")
    print(f"Patch Size: {config['patch_size']}")
    print("="*80)
    
    X, y = load_dataset(data_path, dataset_name, config)
    print(f"Original shape: {X.shape}")
    
    X_pca = applyPCA(X, config['pca_components'])
    print(f"After PCA: {X_pca.shape}")
    
    X_patches, y_labels = createImageCubes(X_pca, y, windowSize=config['patch_size'])
    print(f"Patches: {X_patches.shape}, Samples: {len(y_labels)}")
    
    n_classes = len(np.unique(y_labels))
    print(f"Classes: {n_classes}")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_patches, y_labels, test_size=0.25, stratify=y_labels, random_state=SEED
    )
    S, _, L = X_train.shape[1], X_train.shape[2], X_train.shape[3]
    
    train_dataset = torch.utils.data.TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
    
    # ==================== BASELINE ====================
    print("\n--- Training Baseline (No Compression) ---")
    baseline = FullySelfAdaptiveHybridSN(n_classes, S, L, compress_layers=[]).to(DEVICE)
    baseline_start = time.time()
    baseline = train_two_stage(baseline, train_loader, X_test_t, y_test_t,
                               epochs_stage1=ep_baseline, epochs_stage2=0, final_lambda=0.0)
    baseline_train_time = time.time() - baseline_start
    
    baseline_metrics = evaluate_full(baseline, X_test_t, y_test_t)
    baseline_params = sum(p.numel() for p in baseline.parameters())
    baseline_size_mb = baseline_params * 4 / (1024 * 1024)
    
    # ==================== COMPRESSED ====================
    print("\n--- Training Self-Adaptive Compressed Model ---")
    compress_candidates = ['conv3d_1', 'conv3d_2', 'conv3d_3', 'conv2d', 'fc1', 'fc2']
    student = FullySelfAdaptiveHybridSN(n_classes, S, L, compress_layers=compress_candidates, gate_init=0.0).to(DEVICE)
    student_start = time.time()
    student = train_two_stage(student, train_loader, X_test_t, y_test_t,
                              epochs_stage1=120, epochs_stage2=ep_stage2, final_lambda=1.5, lr=0.0005)
    student_train_time = time.time() - student_start
    
    # Get final learned K before compression
    final_learned_K = student.get_learned_K()
    
    print("\n--- Finalizing Compression ---")
    student.finalize_compression(gate_threshold=0.4)
    
    # Fine-tune
    print("\n--- Fine-tuning Compressed Model ---")
    opt_ft = optim.Adam(student.parameters(), lr=0.0001, weight_decay=1e-5)
    for epoch in range(ep_fine):
        student.train()
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt_ft.zero_grad()
            out = student(xb)
            loss = F.cross_entropy(out, yb)
            loss.backward()
            opt_ft.step()
            total_loss += loss.item()
        if (epoch+1) % 20 == 0:
            val_acc = evaluate_full(student, X_test_t, y_test_t)['accuracy']
            print(f"  FT Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, val_acc={val_acc:.4f}")
    
    compressed_metrics = evaluate_full(student, X_test_t, y_test_t)
    compressed_params = sum(p.numel() for p in student.parameters())
    compressed_size_mb = compressed_params * 4 / (1024 * 1024)
    reduction = (1 - compressed_params / baseline_params) * 100
    acc_change = (compressed_metrics['accuracy'] - baseline_metrics['accuracy']) * 100
    inference_change = (compressed_metrics['inference_time_ms'] - baseline_metrics['inference_time_ms']) / baseline_metrics['inference_time_ms'] * 100
    
    # Print final comparison for this dataset
    print("\n" + "="*80)
    print(f"FINAL RESULTS FOR {dataset_name}")
    print("="*80)
    print(f"\n{'Metric':<20} {'Baseline':<18} {'Compressed':<18} {'Change':<12}")
    print("-"*70)
    print(f"{'Accuracy':<20} {baseline_metrics['accuracy']:<18.4f} {compressed_metrics['accuracy']:<18.4f} {acc_change:>+11.2f}%")
    print(f"{'Precision':<20} {baseline_metrics['precision']:<18.4f} {compressed_metrics['precision']:<18.4f}")
    print(f"{'Recall':<20} {baseline_metrics['recall']:<18.4f} {compressed_metrics['recall']:<18.4f}")
    print(f"{'F1 Score':<20} {baseline_metrics['f1']:<18.4f} {compressed_metrics['f1']:<18.4f}")
    print(f"{'Parameters':<20} {baseline_params:<18,} {compressed_params:<18,} {reduction:>+11.1f}%")
    print(f"{'Model Size (MB)':<20} {baseline_size_mb:<18.2f} {compressed_size_mb:<18.2f} {((1-compressed_size_mb/baseline_size_mb)*100):>+11.1f}%")
    print(f"{'Inference (ms)':<20} {baseline_metrics['inference_time_ms']:<18.2f} {compressed_metrics['inference_time_ms']:<18.2f} {inference_change:>+11.1f}%")
    print(f"{'Training Time (s)':<20} {baseline_train_time:<18.2f} {student_train_time:<18.2f} {((student_train_time/baseline_train_time-1)*100):>+11.1f}%")
    
    print(f"\n📈 LEARNED K VALUES PER LAYER:")
    for layer, k in final_learned_K.items():
        print(f"    {layer}: K={k}")
    
    return {
        'dataset': dataset_name,
        'learned_K': final_learned_K,
        'baseline_params': baseline_params,
        'compressed_params': compressed_params,
        'reduction': reduction,
        'baseline_size_mb': baseline_size_mb,
        'compressed_size_mb': compressed_size_mb,
        'baseline_acc': baseline_metrics['accuracy'],
        'compressed_acc': compressed_metrics['accuracy'],
        'acc_change': acc_change,
        'baseline_precision': baseline_metrics['precision'],
        'compressed_precision': compressed_metrics['precision'],
        'baseline_recall': baseline_metrics['recall'],
        'compressed_recall': compressed_metrics['recall'],
        'baseline_f1': baseline_metrics['f1'],
        'compressed_f1': compressed_metrics['f1'],
        'baseline_inf': baseline_metrics['inference_time_ms'],
        'compressed_inf': compressed_metrics['inference_time_ms'],
        'inference_change': inference_change,
        'baseline_train_time': baseline_train_time,
        'compressed_train_time': student_train_time
    }



Using device: cpu


In [2]:
config =  {
        'file': 'Botswana.mat',
        'gt_file': 'Botswana_gt.mat',
        'data_key': 'Botswana',
        'gt_key': 'Botswana_gt',
        'pca_components': 30,
        'patch_size': 5,
        'description': 'Botswana Delta, 14 classes'
    }
data_path = r"Data"
dataset_name = 'Botswana'
run_experiment_on_dataset(data_path, dataset_name, config, 100, 50, 80)


DATASET: Botswana
Description: Botswana Delta, 14 classes
PCA Components: 30
Patch Size: 5
Original shape: (1476, 256, 145)
After PCA: (1476, 256, 30)
Patches: (3248, 5, 5, 30), Samples: 3248
Classes: 14

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0189 | Val Acc: 1.0000
Stage1 Epoch 40/100 | Loss: 0.0051 | Val Acc: 1.0000
Early stopping at epoch 45 (best val_acc=1.0000)

--- Stage 2: Adding compression loss (K evolution will be shown) ---

--- Training Self-Adaptive Compressed Model ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.4161 | Val Acc: 0.9039
Stage1 Epoch 40/120 | Loss: 0.0614 | Val Acc: 0.9951
Stage1 Epoch 60/120 | Loss: 0.0204 | Val Acc: 0.9988
Early stopping at epoch 78 (best val_acc=0.9988)

--- Stage 2: Adding compression loss (K evolution will be shown) ---

📊 Epoch 10/50 | λ=0.276 | Val Acc=0.9963
   Gates: {'conv3d_1': 0.477819561958313, 'conv3d_2':

{'dataset': 'Botswana',
 'learned_K': {'conv3d_1': 2,
  'conv3d_2': 4,
  'conv3d_3': 8,
  'conv2d': 17,
  'fc1': 70,
  'fc2': 35},
 'baseline_params': 889726,
 'compressed_params': 64926,
 'reduction': 92.70269723487905,
 'baseline_size_mb': 3.3940353393554688,
 'compressed_size_mb': 0.24767303466796875,
 'baseline_acc': 1.0,
 'compressed_acc': 0.9901477832512315,
 'acc_change': -0.9852216748768461,
 'baseline_precision': 1.0,
 'compressed_precision': 0.9911069256297836,
 'baseline_recall': 1.0,
 'compressed_recall': 0.987257506241363,
 'baseline_f1': 1.0,
 'compressed_f1': 0.9890647554752262,
 'baseline_inf': 262.76124399737455,
 'compressed_inf': 51.17250199691625,
 'inference_change': -80.52509524676039,
 'baseline_train_time': 62.24113130569458,
 'compressed_train_time': 236.10244941711426}

In [2]:
config =  {
        'file': 'PaviaU.mat',
        'gt_file': 'PaviaU_gt.mat',
        'data_key': 'paviaU',
        'gt_key': 'paviaU_gt',
        'pca_components': 8,
        'patch_size': 5,
        'description': 'Pavia University, 9 classes'
    }
data_path = r"Data"
dataset_name = 'PaviaU'
run_experiment_on_dataset(data_path, dataset_name, config, 100, 4, 200)


DATASET: PaviaU
Description: Pavia University, 9 classes
PCA Components: 8
Patch Size: 5
Original shape: (610, 340, 103)
After PCA: (610, 340, 8)
Patches: (42776, 5, 5, 8), Samples: 42776
Classes: 9

--- Training Baseline (No Compression) ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/100 | Loss: 0.0383 | Val Acc: 0.9870
Stage1 Epoch 40/100 | Loss: 0.0196 | Val Acc: 0.9882
Stage1 Epoch 60/100 | Loss: 0.0103 | Val Acc: 0.9945
Stage1 Epoch 80/100 | Loss: 0.0065 | Val Acc: 0.9937
Stage1 Epoch 100/100 | Loss: 0.0060 | Val Acc: 0.9921

--- Stage 2: Adding compression loss (K evolution will be shown) ---

--- Training Self-Adaptive Compressed Model ---

--- Stage 1: Training without compression loss ---
Stage1 Epoch 20/120 | Loss: 0.1062 | Val Acc: 0.9652
Stage1 Epoch 40/120 | Loss: 0.0509 | Val Acc: 0.9782
Stage1 Epoch 60/120 | Loss: 0.0305 | Val Acc: 0.9857
Stage1 Epoch 80/120 | Loss: 0.0181 | Val Acc: 0.9907
Stage1 Epoch 100/120 | Loss: 0.0136 | Val Acc: 0.9918
S

{'dataset': 'PaviaU',
 'learned_K': {'conv3d_1': 2,
  'conv3d_2': 4,
  'conv3d_3': 9,
  'conv2d': 18,
  'fc1': 72,
  'fc2': 36},
 'baseline_params': 844025,
 'compressed_params': 67168,
 'reduction': 92.04194188560766,
 'baseline_size_mb': 3.2196998596191406,
 'compressed_size_mb': 0.2562255859375,
 'baseline_acc': 0.9921451281092202,
 'compressed_acc': 0.9505330091640172,
 'acc_change': -4.1612118945203,
 'baseline_precision': 0.9915747143914646,
 'compressed_precision': 0.9531896235287513,
 'baseline_recall': 0.9875102783388319,
 'compressed_recall': 0.945880010679039,
 'baseline_f1': 0.9892522178196157,
 'compressed_f1': 0.9490948890291031,
 'baseline_inf': 876.5862286090851,
 'compressed_inf': 182.34306760132313,
 'inference_change': -79.19850190999985,
 'baseline_train_time': 885.3758282661438,
 'compressed_train_time': 1689.2934701442719}